# Module 2: Wednesday Session
## CS 5045: Computation for the Data Sciences
### Data Representation & Number Systems

This notebook accompanies the Wednesday live session for Module 2.  
Run cells top to bottom. Instructor notes appear in **bold** within markdown cells.

Dataset: `data.csv`, World City Climate Data (16 cities)

---
## Setup

**Instructor note:** Before running this cell, ask students: *"Is `import pandas` a computational operation? Which one?"*  
Answer: No; it's setup. The Represent operation begins on the next cell when we actually load data.

In [1]:
import pandas as pd
import math

# Confirm the library loaded
print(f"pandas version: {pd.__version__}")
print("Ready.")

pandas version: 3.0.3
Ready.


---
## REPRESENT: Loading the Dataset

The **Represent** operation: move data from a file on disk into memory so Python can work with it.  
Before this cell, `df` does not exist. After it, we have a 16-row table with 6 columns.

**Pause and ask:** *"What is the data.csv file before we run this cell, from Python's perspective?"*  
Answer: Nothing; it's invisible to the program. `pd.read_csv()` is what brings it into existence in memory.

In [2]:
# REPRESENT: load the climate dataset
df = pd.read_csv("data.csv")

print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} columns")
df

Loaded: 16 rows x 6 columns


,city,country,avg_temp_c,avg_temp_f,avg_rainfall_mm,is_tropical
0,Nairobi,Kenya,17.6,63.7,686,False
1,Bangkok,Thailand,28.3,82.9,1400,True
2,Oslo,Norway,6.8,44.2,763,False
3,São Paulo,Brazil,19.7,67.5,1454,False
4,Phoenix,United States,23.5,74.3,208,False
5,Mumbai,India,27.2,81.0,2167,True
6,Helsinki,Finland,5.2,41.4,612,False
7,Singapore,Singapore,27.0,80.6,2340,True
8,Buenos Aires,Argentina,17.9,64.2,1027,False
9,Lagos,Nigeria,26.8,80.2,1657,True


---
## The Four-Line Audit

Run these four lines every time you load a new dataset.  
They are all **Communicate** operations; we're asking Python to show us information about the data we just loaded.

**Instructor note:** Walk through each output line by line. Point out:
- `object` = pandas word for string (text)
- `float64` = 64-bit floating point (decimal)
- `int64` = 64-bit integer (whole number)
- `bool` = True/False

Ask: *"Why is avg_rainfall_mm int64 but avg_temp_c is float64? Is that correct?"*  
Answer: Rainfall in millimeters is recorded as whole numbers here; temperature can be fractional.

In [3]:
# Audit step 1: shape — did it load completely?
print("=== Shape ===")
print(df.shape)
print()

# Audit step 2: types — are numeric columns numeric?
print("=== Column Types ===")
print(df.dtypes)
print()

# Audit step 3: first rows — does the data look right?
print("=== First 5 Rows ===")
print(df.head())
print()

# Audit step 4: missing values — any nulls to deal with?
print("=== Missing Values ===")
print(df.isna().sum())

=== Shape ===
(16, 6)

=== Column Types ===
city                   str
country                str
avg_temp_c         float64
avg_temp_f         float64
avg_rainfall_mm      int64
is_tropical           bool
dtype: object

=== First 5 Rows ===
        city        country  avg_temp_c  avg_temp_f  avg_rainfall_mm  \
0    Nairobi          Kenya        17.6        63.7              686   
1    Bangkok       Thailand        28.3        82.9             1400   
2       Oslo         Norway         6.8        44.2              763   
3  São Paulo         Brazil        19.7        67.5             1454   
4    Phoenix  United States        23.5        74.3              208   

   is_tropical  
0        False  
1         True  
2        False  
3        False  
4        False  

=== Missing Values ===
city               0
country            0
avg_temp_c         0
avg_temp_f         0
avg_rainfall_mm    0
is_tropical        0
dtype: int64


---
## What a Broken Type Looks Like

**Instructor note:** This cell deliberately creates a type error. The goal is to show students  
what failure looks like *downstream*: the load succeeds, but analysis breaks.  
This is why the audit matters.

**Pause and ask after running:** *"At what point did this fail: when we loaded the data, or when we tried to analyze it?"*  
Answer: At analysis. The load always succeeds; the type error is silent until you try to do math.

In [ ]:
# Simulate: avg_temp_c was loaded as string (e.g., due to a stray quote in the CSV)
df_broken = df.copy()
df_broken["avg_temp_c"] = df_broken["avg_temp_c"].astype(str)  # force to string

print("Column type after forced conversion:")
print(df_broken["avg_temp_c"].dtype)
print()
print("Values look fine:")
print(df_broken["avg_temp_c"].head(3).tolist())
print()

# Now try to compute the mean — this will fail
try:
    mean_temp = df_broken["avg_temp_c"].mean()
    print(f"Mean temperature: {mean_temp}")
except TypeError as e:
    print(f"TypeError: {e}")
    print()
    print("Fix: df['avg_temp_c'] = df['avg_temp_c'].astype(float)")
    df_broken["avg_temp_c"] = df_broken["avg_temp_c"].astype(float)
    print(f"Fixed! Mean temperature: {df_broken['avg_temp_c'].mean():.1f}°C")

---
## TRANSFORM: Verifying Two Representations of Temperature

The dataset has both `avg_temp_c` (Celsius) and `avg_temp_f` (Fahrenheit).  
They represent the **same physical quantity**, just encoded differently.

The conversion formula: **F = C × 9/5 + 32**

This is a **Transform** operation: we take one representation and produce another,  
then compare it to what's already in the dataset as a data quality check.

**Pause and ask:** *"If the rounding error is 0.1 for some cities, is that a data quality problem or a floating-point issue?"*  
Answer: 0.1°F is rounding in the original data (values were rounded to one decimal). Not a bug.

In [ ]:
# TRANSFORM: recompute Fahrenheit from Celsius
df["temp_f_check"]   = df["avg_temp_c"] * 9 / 5 + 32
df["rounding_error"] = (df["temp_f_check"] - df["avg_temp_f"]).abs()

# COMMUNICATE: display the comparison
comparison = df[["city", "avg_temp_c", "avg_temp_f", "temp_f_check", "rounding_error"]].copy()
comparison["temp_f_check"]   = comparison["temp_f_check"].round(2)
comparison["rounding_error"] = comparison["rounding_error"].round(4)
print(comparison.to_string(index=False))

print()
print(f"Maximum rounding error: {df['rounding_error'].max():.4f}°F")
print("Conclusion: columns are consistent — small errors are floating-point rounding.")

# Clean up the check columns
df.drop(columns=["temp_f_check", "rounding_error"], inplace=True)

---
## Bool as Arithmetic: AGGREGATE with a Boolean Column

In Python, `True` is treated as `1` and `False` as `0` in arithmetic contexts.  
That means we can **count True values** using `sum()`.

**Instructor note:** This surprises many students. Show the sum, then confirm with value_counts().

**Pause and ask:** *"What would the mean of a boolean column tell you?"*  
Answer: The proportion of True values; e.g., 0.375 means 37.5% of cities are tropical.

In [ ]:
# AGGREGATE: count tropical and non-tropical cities
tropical_count     = df["is_tropical"].sum()    # True=1, False=0
nontropical_count  = (~df["is_tropical"]).sum()  # ~ inverts booleans
tropical_fraction  = df["is_tropical"].mean()   # proportion

print(f"Tropical cities:     {int(tropical_count)}")
print(f"Non-tropical cities: {int(nontropical_count)}")
print(f"Fraction tropical:   {tropical_fraction:.1%}")
print()

# Confirm with value_counts()
print("=== value_counts() breakdown ===")
print(df["is_tropical"].value_counts())

---
## Floating-Point Precision: A Demonstration

A well-known Python quirk, not a bug in your code.

In [ ]:
# The classic floating-point surprise
result = 0.1 + 0.2
print(f"0.1 + 0.2 = {result}")
print(f"0.1 + 0.2 == 0.3 ? {result == 0.3}")
print()

# The correct way to compare floats
tolerance = 1e-9
print(f"abs(result - 0.3) < {tolerance} ? {abs(result - 0.3) < tolerance}")
print(f"math.isclose(result, 0.3) ? {math.isclose(result, 0.3)}")
print()
print("Rule: never use == to compare two floats. Use math.isclose() or a tolerance check.")

---
## Connecting to the Assignment

This week's assignment mirrors exactly what we just did:

1. **Represent:** Load your dataset with `pd.read_csv()`
2. **Communicate:** Run the four-line audit (shape, dtypes, head, isna)
3. **Transform:** Fix any type errors with `.astype()`
4. **Transform + Communicate:** Verify one numeric relationship between two columns

The assignment scaffold is `hw2.py` (download from Canvas along with `data.csv`). Use `example.py` in the `Code/` folder as a reference for how the complete pipeline looks; you can adapt from it, but submit `hw2.py`.

**Questions to bring to Canvas or office hours:**
- My numeric column is showing as `object`; how do I find out why?
- My dataset has only one numeric column; what relationship can I verify?
- One city has a NaN in rainfall; do I need to handle that before computing the mean?

See you next week for Module 3: Conditionals and Control Flow.